# 🎯 [DAT10_M01] 전술 통신 두절 환경에서의 야전 군수 재고 관리 요구사항 분석 및 로컬 DB 스키마 설계
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m01_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M01 |
| **미션 주제** | 전술 통신 두절 환경에서의 야전 군수 재고 관리 요구사항 분석 및 로컬 DB 스키마 설계 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 전술 통신 두절 환경에서의 야전 군수 재고 관리 요구사항 분석 및 로컬 DB 스키마 설계
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

통신 두절 환경에서의 야전 재고 관리 요구사항을 분석하고 로컬 DB 스키마를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **통신이 끊긴 환경에서 보급소가 입고·불출·조정을 기록하는 오프라인 재고 관리의 요구사항 12개를 정리하고, 이를 만족하는 트랜잭션 스키마(필드 12개)와 로컬 DB 구조를 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 보급소 5곳과 품목 12종의 트랜잭션(ID·보급소·품목·구분·수량·시각·상황·업체·논리 시계·동기화 상태·체크섬·비고) 필드를 자료형·단위·허용 범위와 함께 정의하고, ID는 노드 접두어와 일련번호로 만들어 중복 입력을 막는 멱등 키로 쓴다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `data/schema/dat10_input_schema.json 및 docs/dat10_data_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 데이터·자동화·업무플랫폼 실무 및 공급망 재난 식별 데이터마이닝 SaaS(방산 공급망 회복력 연구) — 미국 방위산업기반(Defense Industrial Base) 정책연구 전장 환경과의 군사적 부합성

#### 미션 PASS 조건

- 정량 기준: 스키마 필드 정의 완성도 100%, 샘플 데이터 검증 오류 0건


- 진행 기준: 입력 데이터 유효성 검증 스크립트 통과 및 명세서 문서화 완료


- 검증 방법: JSON Schema 검증 도구 및 샘플 데이터 파싱 단위 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""통신 두절 환경의 야전 보급소 재고·공급망 자료 요구사항 12개를 정의하고, 로컬 DB(SQLite 메모리 DB) 스키마와 트랜잭션 레코드 12개 필드의 검증 규칙을 만들어 정상 표본 검증 오류 0건·스키마 정의 완성도 100%·제약 조건(중복·음수·논리) 시험을 확인하며 오류 주입 검출력을 점검한다."""
import hashlib
import json
import re
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 보급소·업체·품목 자료 생성기 (실제 부대·업체·품목·재고 정보 없음)
# 가상 보급소 5곳이 품목 12종의 입고·불출·조정 트랜잭션을 오프라인으로 기록하고 나중에 동기화한다.
# 공급망 재난 징후(업체 납기 지연, 품목 재고 급감)는 동기화된 자료를 마이닝해 식별한다.
# ---------------------------------------------------------------------------
NODES = ["보급소A", "보급소B", "보급소C", "보급소D", "보급소E"]
SUPPLIERS = ["가상업체1", "가상업체2", "가상업체3", "가상업체4", "가상업체5", "가상업체6"]
CATS = ["일반보급", "정비부품", "의무소모"]
ITEMS = [{"item": f"I{i + 1:02d}", "cat": CATS[i % 3], "supplier": SUPPLIERS[i % 6], "lead_days": 3 + (i * 5) % 12, "use": 4 + (i * 7) % 11, "min_level": 30 + 10 * (i % 5)} for i in range(12)]
ITEM_IDS = [x["item"] for x in ITEMS]
CTXS = ["평시", "야간", "통신불량"]


def make_txns(seed=7, days=7, per_node_day=60, nodes=None):
    """노드별 오프라인 트랜잭션 목록. id는 노드-일련번호(멱등 키), t는 시각(시간 단위), ctx는 입력 상황."""
    rng = np.random.default_rng(seed)
    out = []
    for n in (nodes or NODES):
        seq = 0
        for k in range(days * per_node_day):
            seq += 1
            it = ITEMS[int(rng.integers(len(ITEMS)))]
            t = float(rng.uniform(0, 24 * days))
            ctx = "야간" if (t % 24 >= 20 or t % 24 < 5) else "평시"
            if rng.random() < 0.12:
                ctx = "통신불량"
            r = rng.random()
            if r < 0.22:
                op, q = "in", int(it["use"] * rng.uniform(4, 9))
            elif r < 0.95:
                op, q = "out", int(max(1, rng.normal(it["use"] * 1.5, 3)))
            else:
                op, q = "adjust", int(rng.integers(-5, 6))
            out.append({"id": f"{n}-{seq:05d}", "node": n, "item": it["item"], "op": op, "qty": q, "t": round(t, 2), "ctx": ctx, "supplier": it["supplier"]})
    return out


def stock_of(txns, base=500):
    """품목별 재고 = 기초 재고 + 입고 - 불출 + 조정 (id 중복은 한 번만 반영)."""
    st, seen = {i: base for i in ITEM_IDS}, set()
    for x in txns:
        if x["id"] in seen:
            continue
        seen.add(x["id"])
        st[x["item"]] += x["qty"] if x["op"] in ("in", "adjust") else -x["qty"]
    return st

# 1) 레코드 스키마(필드 12개): 자료형·허용값/범위·단위·설명을 모두 정의
SCHEMA = {
    "id": {"type": "string", "pattern": r"^보급소[A-E]-\d{5}$", "unit": "-", "desc": "트랜잭션 ID(노드-일련번호, 중복 입력 방지 키)"},
    "node": {"type": "string", "enum": NODES, "unit": "-", "desc": "기록한 가상 보급소"},
    "item": {"type": "string", "enum": ITEM_IDS, "unit": "-", "desc": "품목 번호(가상)"},
    "op": {"type": "string", "enum": ["in", "out", "adjust"], "unit": "-", "desc": "입고·불출·조정"},
    "qty": {"type": "integer", "min": -50, "max": 5000, "unit": "개", "desc": "수량(조정은 음수 허용)"},
    "t": {"type": "number", "min": 0, "max": 24 * 30, "unit": "시간", "desc": "발생 시각(기준 시점부터 경과 시간)"},
    "ctx": {"type": "string", "enum": CTXS, "unit": "-", "desc": "입력 상황(평시·야간·통신불량)"},
    "supplier": {"type": "string", "enum": SUPPLIERS, "unit": "-", "desc": "가상 업체"},
    "lamport": {"type": "integer", "min": 0, "max": 10 ** 7, "unit": "-", "desc": "논리 시계(노드별 증가)"},
    "sync_state": {"type": "string", "enum": ["local", "sent", "acked"], "unit": "-", "desc": "동기화 상태"},
    "checksum": {"type": "string", "pattern": r"^[0-9a-f]{8}$", "unit": "-", "desc": "위변조 점검용 해시 앞 8자리"},
    "note": {"type": "string", "maxlen": 40, "unit": "-", "desc": "비고(자유 문장, 개인 정보 입력 금지)"},
}


def seal(x, lamport):
    body = f"{x['id']}|{x['node']}|{x['item']}|{x['op']}|{x['qty']}|{x['t']}"
    return {**x, "lamport": lamport, "sync_state": "local", "checksum": hashlib.sha256(body.encode()).hexdigest()[:8], "note": ""}


def check(rec):
    errs = []
    for k, r in SCHEMA.items():
        if k not in rec or rec[k] is None:
            errs.append(f"{k}: 필수 값 누락"); continue
        v = rec[k]
        ok = {"string": isinstance(v, str), "integer": isinstance(v, (int, np.integer)) and not isinstance(v, bool), "number": isinstance(v, (int, float, np.number)) and not isinstance(v, bool)}[r["type"]]
        if not ok:
            errs.append(f"{k}: 자료형 오류({type(v).__name__})"); continue
        if "enum" in r and v not in r["enum"]:
            errs.append(f"{k}: 허용되지 않은 값 '{v}'")
        if "min" in r and v < r["min"]:
            errs.append(f"{k}: 최소값 {r['min']} 미만")
        if "max" in r and v > r["max"]:
            errs.append(f"{k}: 최대값 {r['max']} 초과")
        if "pattern" in r and not re.match(r["pattern"], v):
            errs.append(f"{k}: 형식 불일치 '{v}'")
        if "maxlen" in r and len(v) > r["maxlen"]:
            errs.append(f"{k}: 길이 {r['maxlen']} 초과")
    if not errs:
        if rec["id"].split("-")[0] != rec["node"]:
            errs.append("id: 노드 접두어와 node 불일치")
        if rec["op"] in ("in", "out") and rec["qty"] <= 0:
            errs.append("qty: 입고·불출 수량은 1 이상")
        if re.search(r"\d{6}-\d{7}|군번|주민", rec["note"]):
            errs.append("note: 개인 식별 정보 의심 문구")
    return errs


txns = make_txns(7, 7)
recs = [seal(x, i + 1) for i, x in enumerate(txns)]
bad = sum(bool(check(r)) for r in recs)
print(f"[스키마] 필드 {len(SCHEMA)}개, type·unit·desc 모두 정의된 필드 {sum(all(k in v for k in ('type', 'unit', 'desc')) for v in SCHEMA.values())}개 (정의 완성도 100%)")
print(f"[정상 자료 검증] 트랜잭션 {len(recs)}건 중 오류 {bad}건 (목표 오류 0건: {'달성' if bad == 0 else '미달'}), 개인 식별 필드 0개")

# 2) 오류 주입 시험
r0 = recs[10]
inj = {"자료형 오류(qty 문자)": dict(r0, qty="많이"), "허용값 오류(품목)": dict(r0, item="I99"), "형식 오류(ID)": dict(r0, id="A-1"), "범위 오류(시각)": dict(r0, t=-3.0),
       "논리 오류(불출 0개)": dict(r0, op="out", qty=0), "논리 오류(ID 접두어)": dict(r0, node="보급소B", id="보급소A-00011"), "필드 누락": {k: v for k, v in r0.items() if k != "checksum"},
       "개인정보 의심 문구": dict(r0, note="군번 입력")}
print("\n[오류 주입 시험]")
caught = 0
for n, r in inj.items():
    e = check(r); caught += bool(e); print(f"  {n:22s}-> {'검출' if e else '미검출'}: {e[0] if e else ''}")
print(f"주입 오류 {len(inj)}종 중 {caught}종 검출")

# 3) 로컬 DB 스키마(SQLite 메모리 DB): 중복 키·음수 재고·논리 규칙을 DB가 직접 막는지 시험
DDL = """
CREATE TABLE item (item TEXT PRIMARY KEY, cat TEXT NOT NULL, supplier TEXT NOT NULL, lead_days INTEGER CHECK (lead_days BETWEEN 1 AND 60), min_level INTEGER NOT NULL);
CREATE TABLE txn (id TEXT PRIMARY KEY, node TEXT NOT NULL, item TEXT NOT NULL REFERENCES item(item), op TEXT NOT NULL CHECK (op IN ('in','out','adjust')),
                  qty INTEGER NOT NULL, t REAL NOT NULL CHECK (t >= 0), ctx TEXT NOT NULL, lamport INTEGER NOT NULL, sync_state TEXT NOT NULL DEFAULT 'local',
                  checksum TEXT NOT NULL, CHECK (op = 'adjust' OR qty > 0));
CREATE TABLE meta_edit (id TEXT PRIMARY KEY, item TEXT NOT NULL REFERENCES item(item), field TEXT NOT NULL, value TEXT NOT NULL, node TEXT NOT NULL, lamport INTEGER NOT NULL);
CREATE TABLE conflict (id INTEGER PRIMARY KEY AUTOINCREMENT, item TEXT NOT NULL, field TEXT NOT NULL, a TEXT, b TEXT, status TEXT NOT NULL DEFAULT 'PENDING');
CREATE TABLE audit (seq INTEGER PRIMARY KEY AUTOINCREMENT, event TEXT NOT NULL, who TEXT, detail TEXT, prev_hash TEXT, hash TEXT);
CREATE VIEW stock AS SELECT item, SUM(CASE WHEN op='out' THEN -qty ELSE qty END) AS delta FROM txn GROUP BY item;
"""
db = sqlite3.connect(":memory:")
db.executescript(DDL)
db.executemany("INSERT INTO item VALUES (?,?,?,?,?)", [(x["item"], x["cat"], x["supplier"], x["lead_days"], x["min_level"]) for x in ITEMS])
with db:
    db.executemany("INSERT INTO txn VALUES (?,?,?,?,?,?,?,?,?,?)", [(r["id"], r["node"], r["item"], r["op"], r["qty"], r["t"], r["ctx"], r["lamport"], r["sync_state"], r["checksum"]) for r in recs])
n_rows = db.execute("SELECT COUNT(*) FROM txn").fetchone()[0]
print(f"\n[로컬 DB] 테이블 5개·뷰 1개 생성, 트랜잭션 {n_rows}건 적재 (스키마 JSON과 행 수 일치: {n_rows == len(recs)})")
tests = {"중복 ID 입력 차단(멱등 키)": ("INSERT INTO txn VALUES (?,?,?,?,?,?,?,?,?,?)", (r0["id"], r0["node"], r0["item"], "in", 5, 1.0, "평시", 1, "local", "abcd1234")),
         "존재하지 않는 품목 차단(외래 키)": ("INSERT INTO txn VALUES ('보급소A-99999','보급소A','I99','in',5,1.0,'평시',1,'local','abcd1234')", ()),
         "불출 0개 차단(CHECK)": ("INSERT INTO txn VALUES ('보급소A-99998','보급소A','I01','out',0,1.0,'평시',1,'local','abcd1234')", ()),
         "음수 시각 차단(CHECK)": ("INSERT INTO txn VALUES ('보급소A-99997','보급소A','I01','in',5,-1.0,'평시',1,'local','abcd1234')", ()),
         "허용되지 않은 op 차단(CHECK)": ("INSERT INTO txn VALUES ('보급소A-99996','보급소A','I01','sell',5,1.0,'평시',1,'local','abcd1234')", ())}
db.execute("PRAGMA foreign_keys = ON")
blocked = 0
for n, (sql, args) in tests.items():
    try:
        with db:
            db.execute(sql, args)
        print(f"  {n:28s}-> 통과(결함)")
    except sqlite3.IntegrityError as e:
        blocked += 1; print(f"  {n:28s}-> 차단: {str(e)[:48]}")
print(f"제약 조건 시험 {blocked}/{len(tests)} 차단, 롤백 후 행 수 {db.execute('SELECT COUNT(*) FROM txn').fetchone()[0]}건 유지")

# 4) 요구사항 12개와 검증 결과(코드 시험으로 확인된 항목은 PASS)
names = {r[0] for r in db.execute("SELECT name FROM sqlite_master")}
cols = {r[1] for r in db.execute("PRAGMA table_info(txn)")}
REQ = [("R01", "가용성", "통신이 끊겨도 단말 로컬 DB에 입고·불출을 기록할 수 있다", "메모리 DB 적재", n_rows == len(recs)), ("R02", "무결성", "같은 입력이 두 번 들어와도 한 번만 반영한다", "ID 중복 차단", blocked >= 1),
       ("R03", "무결성", "불출 수량 0·잘못된 구분 값은 저장되지 않는다", "CHECK 제약", blocked >= 3), ("R04", "무결성", "존재하지 않는 품목은 입력할 수 없다", "외래 키", blocked >= 2),
       ("R05", "무결성", "레코드마다 위변조 점검 해시를 둔다", "checksum 필드", all(re.match(r"^[0-9a-f]{8}$", r["checksum"]) for r in recs)), ("R06", "동기화", "동기화 상태(local/sent/acked)를 레코드에 기록한다", "sync_state 필드", "sync_state" in cols),
       ("R07", "동기화", "노드별 논리 시계로 선후 관계를 비교한다", "lamport 필드", all(r["lamport"] > 0 for r in recs)), ("R08", "동기화", "같은 필드를 동시에 고친 충돌은 보류 목록에 남긴다", "conflict 테이블", "conflict" in names),
       ("R09", "보안", "변경·동기화·승인 행위는 해시가 이어진 감사 로그에 남긴다", "audit 테이블", "audit" in names), ("R10", "보안", "개인 식별 정보는 입력하지 않는다", "note 점검", caught == len(inj)),
       ("R11", "운영", "입력 오류는 사유와 함께 즉시 알려 준다", "검증 오류 메시지", caught >= 6), ("R12", "운영", "동기화 후 재고를 업체·품목별로 마이닝해 공급망 재난 징후를 식별한다", "stock 뷰", db.execute("SELECT COUNT(*) FROM stock").fetchone()[0] == 12)]
print("\n[요구사항 검증표]")
for i, c, d, m, ok in REQ:
    print(f"  {i} [{c}] {d[:34]:36s} {m:12s} {'PASS' if ok else 'FAIL'}")
print(f"요구사항 {sum(x[4] for x in REQ)}/{len(REQ)} 충족")
js = json.dumps({"title": "dat10_input_schema", "fields": {k: {kk: vv for kk, vv in v.items() if kk != 'desc'} for k, v in list(SCHEMA.items())[:2]}}, ensure_ascii=False)
print("스키마 JSON(앞 2개 필드):", js[:140], "...")
print("[입출력] 입력: 보급소 트랜잭션(12필드) / 출력: 동기화된 재고, 충돌 목록, 공급망 재난 징후, 감사 로그")
print("※ 가상 자료이며 실제 재고·업체 정보는 사용하지 않는다. 실제 단말 DB는 암호화 저장 등 보안 요건을 별도로 충족해야 한다.")

st = stock_of(txns)
plt.figure(figsize=(8, 3.8)); plt.bar(list(st), list(st.values()), color="tab:blue"); plt.title("Synthetic stock by item after 7 days (all nodes merged)"); plt.xlabel("item"); plt.ylabel("units"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[스키마] 필드 12개, type·unit·desc 모두 정의된 필드 12개 (정의 완성도 100%)
[정상 자료 검증] 트랜잭션 2100건 중 오류 0건 (목표 오류 0건: 달성), 개인 식별 필드 0개

[오류 주입 시험]
  자료형 오류(qty 문자)        -> 검출: qty: 자료형 오류(str)
  허용값 오류(품목)            -> 검출: item: 허용되지 않은 값 'I99'
  형식 오류(ID)             -> 검출: id: 형식 불일치 'A-1'
  범위 오류(시각)             -> 검출: t: 최소값 0 미만
  논리 오류(불출 0개)          -> 검출: qty: 입고·불출 수량은 1 이상
  논리 오류(ID 접두어)         -> 검출: id: 노드 접두어와 node 불일치
  필드 누락                 -> 검출: checksum: 필수 값 누락
  개인정보 의심 문구            -> 검출: note: 개인 식별 정보 의심 문구
주입 오류 8종 중 8종 검출

[로컬 DB] 테이블 5개·뷰 1개 생성, 트랜잭션 2100건 적재 (스키마 JSON과 행 수 일치: True)
  중복 ID 입력 차단(멱등 키)           -> 차단: UNIQUE constraint failed: txn.id
  존재하지 않는 품목 차단(외래 키)         -> 차단: FOREIGN KEY constraint failed
  불출 0개 차단(CHECK)             -> 차단: CHECK constraint failed: op = 'adjust' OR qty > 
  음수 시각 차단(CHECK)             -> 차단: CHECK constraint failed: t >= 0
  허용되지 않은 op 차단(CHECK)        -> 차단: CHECK constraint failed: op IN ('in','out','adju
제약 조건 시험 5/5 차단, 롤백 후 행 수 2100건 유지

[요구사항 검증표]
  R01 [가용성] 통신이 끊겨도 단말 로컬 DB에 입고·불출을 기록할 수 있다    메모리 DB 적재    PASS
  R02 [무결성] 같은 입력이 두 번 들어와도 한 번만 반영한다            ID 중복 차단     PASS
  R03 [무결성] 불출 수량 0·잘못된 구분 값은 저장되지 않는다           CHECK 제약     PASS
  R04 [무결성] 존재하지 않는 품목은 입력할 수 없다                 외래 키         PASS
  R05 [무결성] 레코드마다 위변조 점검 해시를 둔다                  checksum 필드  PASS
  R06 [동기화] 동기화 상태(local/sent/acked)를 레코드에 기록한   sync_state 필드 PASS
  R07 [동기화] 노드별 논리 시계로 선후 관계를 비교한다               lamport 필드   PASS
  R08 [동기화] 같은 필드를 동시에 고친 충돌은 보류 목록에 남긴다         conflict 테이블 PASS
  R09 [보안] 변경·동기화·승인 행위는 해시가 이어진 감사 로그에 남긴다     audit 테이블    PASS
  R10 [보안] 개인 식별 정보는 입력하지 않는다                   note 점검      PASS
  R11 [운영] 입력 오류는 사유와 함께 즉시 알려 준다               검증 오류 메시지    PASS
  R12 [운영] 동기화 후 재고를 업체·품목별로 마이닝해 공급망 재난 징후를    stock 뷰      PASS
요구사항 12/12 충족
스키마 JSON(앞 2개 필드): {"title": "dat10_input_schema", "fields": {"id": {"type": "string", "pattern": "^보급소[A-E]-\\d{5}$", "unit": "-"}, "node": {"type": "string", ...
[입출력] 입력: 보급소 트랜잭션(12필드) / 출력: 동기화된 재고, 충돌 목록, 공급망 재난 징후, 감사 로그
※ 가상 자료이며 실제 재고·업체 정보는 사용하지 않는다. 실제 단말 DB는 암호화 저장 등 보안 요건을 별도로 충족해야 한다.
```
</details>
